# Protego — Training Demo

This notebook trains a **Privacy Protection Texture (PPT)** for one or two protectees on the preprocessed FaceScrub subset, then runs the cross-protectee retrieval evaluation.

For a real run use `python train.py` (full pool, all protectees). Here we use a reduced configuration (fewer epochs / protectees) so the notebook finishes quickly.

Prerequisites: `bash setup_quick.sh`, the `protego` environment, and `python -m tools.download_assets`.

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import warnings
warnings.filterwarnings("ignore")
import math

import torch
from omegaconf import OmegaConf

from protego import BASE_PATH
from protego.protego_train import train_protego_mask
from protego.run_exp import run
from protego.FacialRecognition import BASIC_POOL

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 1. Configuration

We hold out `ir50_adaface_casia` as the unseen evaluation model and train on the rest of the basic pool. `epoch_num` is reduced for the demo; increase it (e.g. 100) for full-strength PPTs.

In [ ]:
configs = {
    'global_random_seed': 42,
    'device': str(device),
    'exp_name': 'demo_train',

    # Training data
    'train_portion': 0.6,
    'uv_gen_align_ldmk': False,
    'uv_gen_batch': 8,
    'need_cropping': False,
    'fd_name': 'mtcnn',
    'crop_loosen': 1.,
    'shuffle': False,

    # Training configs (reduced for the demo)
    'three_d': True,
    'epoch_num': 20,
    'batch_size': 16,
    'epsilon': 16 / 255.,
    'min_ssim': 0.95,
    'learning_rate': 0.01 * (16 / 255.),
    'mask_size': 224,
    'mask_random_seed': 114,
    'bin_mask': True,
    'train_fr_names': [n for n in BASIC_POOL if n != 'ir50_adaface_casia'],

    # Eval configs
    'mask_name': ['demo_train', 'univ_mask.npy'],
    'eval_db': 'face_scrub',
    'eval_fr_names': ['ir50_adaface_casia'],
    'save_univ_mask': True,
    'visualize_interval': 10,
    'query_portion': 0.5,
    'vis_eval': True,
    'lpips_backbone': 'vgg',
}
cfgs = OmegaConf.create(configs)
torch.manual_seed(cfgs.global_random_seed)

## 2. Build the train / eval splits

For the demo we only use the first two protectees. Drop the `[:2]` slice to train on everyone.

In [ ]:
def list_face_images(folder):
    return sorted([os.path.join(folder, f) for f in os.listdir(folder)
                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')) and not f.startswith(('.', '_'))])

train_data_path = os.path.join(BASE_PATH, 'face_db', cfgs.eval_db)
protectees = sorted([n for n in os.listdir(train_data_path) if not n.startswith(('.', '_'))])[:2]
data = {}
for protectee in protectees:
    imgs = list_face_images(os.path.join(train_data_path, protectee))
    train_num = math.floor(len(imgs) * cfgs.train_portion)
    data[protectee] = {'train': imgs[:train_num], 'eval': imgs[train_num:]}
print('Protectees:', protectees)

## 3. Train + evaluate

`run(..., mode='train', ...)` trains a PPT per protectee (saved to `experiments/demo_train/<protectee>/univ_mask.npy`), then runs the cross-protectee retrieval evaluation and writes `eval_res_*.yaml` next to each mask.

In [ ]:
run(cfgs, mode='train', data=data, train=train_protego_mask)
print('Done. Trained PPTs are under experiments/demo_train/<protectee>/univ_mask.npy')